# Esports Devs API

In [ ]:
# Find leaf keys function

from typing import Any, List, Tuple

def _contains_dict(obj: Any) -> bool:
    """Visszaadja True-t, ha obj maga dict, vagy (rekurzívan) tartalmaz dict-et (pl. listában)."""
    if isinstance(obj, dict):
        return True
    if isinstance(obj, list):
        for el in obj:
            if _contains_dict(el):
                return True
    return False

def find_leaf_keys(data: dict, *, sep: str = ".") -> List[str]:
    """
    Visszaadja a levélkulcsok teljes útvonalait (pl. "a.b.c") azoknak a kulcsoknak,
    amelyek értéke nem dict és nem tartalmaz dict-et listában sem.
    """
    leaves: List[str] = []

    def _recurse(obj: Any, path: List[str]):
        if isinstance(obj, dict):
            for k, v in obj.items():
                new_path = path + [str(k)]
                # ha az érték dict-et (vagy listában dict-et) tartalmaz -> megyünk tovább
                if _contains_dict(v):
                    _recurse(v, new_path)
                else:
                    # ez levél: nem dict és a list sem tartalmaz dict-et
                    leaves.append(sep.join(new_path))
        elif isinstance(obj, list):
            # listát akkor járjuk be, ha szeretnénk megtalálni a benne lévő dict-ek leveleit
            for idx, el in enumerate(obj):
                _recurse(el, path + [f"[{idx}]"])
        else:
            # objektum önmagában (nem dict, nem list): ha path utolsó elemre vonatkozik, már lefutott korábban
            pass

    _recurse(data, [])
    return leaves

# Ha csak a levélkulcs "név"-eket akarod (nem teljes útvonal), ezt használhatod:
def find_leaf_key_names(data: dict) -> List[str]:
    paths = find_leaf_keys(data)
    return [p.split(".")[-1] for p in paths]

In [ ]:
import requests
import os 
import dotenv

dotenv.load_dotenv()

API_KEY = os.getenv('API_KEY')
BASE_URL = "https://esports-devs.p.rapidapi.com"

headers = {
    "x-rapidapi-key": API_KEY,
    "x-rapidapi-host": "esports-devs.p.rapidapi.com"
}

In [ ]:
# Leagues

response = requests.get(
    f"{BASE_URL}/leagues",
    headers=headers,
    params={
        "limit": 10,
        "offset": 0,
        "class_id": "eq.112"
    }
)

leagues = response.json()
print(leagues)

In [ ]:
for league in leagues:
    print(f"{league['name']} ({league['id']})")

In [ ]:
# Tournaments

response = requests.get(
    f"{BASE_URL}/tournaments",
    headers=headers,
    params={
        "league_id": "eq.2175",  # BLAST Premier
        "limit": 10
    }
)
tournaments = response.json()
print(tournaments)

In [ ]:
for tour in tournaments:
    print(f"{tour['name']} ({tour['id']})")

In [ ]:
# Seasons

response = requests.get(
    f"{BASE_URL}/seasons",
    headers=headers,
    params={
        "league_id": "eq.2191",
        "limit": 10
    }
)

seasons = response.json()
print(seasons)

In [ ]:
for season in seasons:
    print(f"{season['name']} ({season['id']})")

In [ ]:
# Matches by season

response = requests.get(
    f"{BASE_URL}/matches",
    headers=headers,
    params={
        "limit": 10,
        "offset": 0,
        "tournament_id": "eq.16810"
        #"season_id": "eq.13044"
    }
)

matches = response.json()
print(matches)

In [ ]:
for key in find_leaf_keys(matches[0]):
    print(key)

In [ ]:
for match in matches:
    print(match['start_time'])
    print(f"{match['home_team_name']} ({match['home_team_id']})" \
          f" vs {match['away_team_name']} ({match['away_team_id']})" \
          f" (match: {match['id']})")
    print(f"{match['home_team_score']['current']}" \
          f" - {match['away_team_score']['current']}\n")

In [ ]:
# Games of match

response = requests.get(
    f"{BASE_URL}/matches-games",
    headers=headers,
    params={
        "limit": 10,
        "offset": 0,
        "match_id": "eq.88423"
    }
)

games = response.json()
print(games)

In [ ]:
for key in find_leaf_keys(games[0]):
    print(key)

In [ ]:
for game in games:
    print(f"{game['map']} ({game['id']})\n" \
          f"{game['home_team_score']['display']} - {game['away_team_score']['display']}\n" \
          f"- Stats: {game['has_statistics']}\n- Rounds: {game['has_rounds']}\n- Lineups: {game['has_lineups']}\n")

In [ ]:
# Team

response = requests.get(
    f"{BASE_URL}/teams",
    headers=headers,
    params={
        "limit": 50,
        "id": "eq.2433"
    }
)

teams = response.json()
print(teams)

In [ ]:
# Odds coverage

response = requests.get(
    f"{BASE_URL}/odds/coverage",
    headers=headers,
    params={
        "limit": 50,
        "match_id": "eq.222076"
    }
)

odds_cov = response.json()
print(odds_cov)

In [ ]:
import pandas as pd

leagues = pd.read_csv("output/leagues.csv")
seasons = pd.read_csv("output/seasons.csv")
matches = pd.read_csv("output/matches.csv")

#display(leagues) # 2104 - blast premier
#display(seasons[seasons.league_id == 2104]) # 22938 - globals final 2020
display(matches[matches.season_id == 22938])

In [ ]:
print(leagues.name.unique())